# Task 1

In [108]:
import pandas as pd

loans_raw = pd.read_csv("../data/loans.csv")
loans = loans_raw.copy()

print(loans.shape)
loans.info()

(623, 9)
<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loan_id      623 non-null    str    
 1   member_id    623 non-null    str    
 2   title        623 non-null    str    
 3   genre        623 non-null    str    
 4   loan_date    623 non-null    str    
 5   return_date  593 non-null    str    
 6   late_fee     623 non-null    str    
 7   returned     623 non-null    str    
 8   notes        0 non-null      float64
dtypes: float64(1), str(8)
memory usage: 43.9 KB


Initial thoughts about potential problems:
* The notes column is empty
* The late_fee column is a string, should be a numeric value
* All the date columns are strings, would be better as datetime
* There are less values in the return_date column than the others, so some values appear to be missing
* Looking at the .csv file, there appears to be duplicate entries
* Looking at the .csv file, the genre column has inconsistent capitalisation
* Looking at the .csv file, the returned column has multiple different yes/no values, would be better as booleans

# Task 2

In [96]:
loans["genre"].value_counts()

genre
Crime              38
Crime              35
crime              33
Romance            32
Children           31
Children's         31
fantasy            29
children           29
 crime             29
CRIME              28
children's         28
history            22
Fantasy            22
Romance            20
 Fantasy           20
romance            19
FANTASY            19
science fiction    19
sci fi             17
Children           16
Sci-fi             16
 history           16
 ROMANCE           15
sci-fi             14
History            14
Science fiction    14
Sci fi             10
History             7
Name: count, dtype: int64

In [109]:
loans["genre"] = loans["genre"].str.strip().str.lower()
loans["genre"].value_counts()

genre
crime              163
fantasy             90
romance             86
children            76
children's          59
history             59
science fiction     33
sci-fi              30
sci fi              27
Name: count, dtype: int64

In [98]:
loans["genre"].unique()

<StringArray>
[        'romance',           'crime',         'fantasy',        'children',
      'children's',         'history',          'sci-fi', 'science fiction',
          'sci fi']
Length: 9, dtype: str

In [110]:
loans["genre"] = loans["genre"].replace("children's", "children")
loans["genre"].value_counts()

loans["genre"] = loans["genre"].replace("sci fi", "science fiction")
loans["genre"].value_counts()

loans["genre"] = loans["genre"].replace("sci-fi", "science fiction")
loans["genre"].value_counts()

loans["genre"] = loans["genre"].str.title()
loans["genre"].value_counts()

genre
Crime              163
Children           135
Fantasy             90
Science Fiction     90
Romance             86
History             59
Name: count, dtype: int64

## One step genre cleaning

In [100]:
# Alternative solution, cleaning genres in one step:
loans["genre"] = (
    loans["genre"]
    .str.strip()
    .str.lower()
    .replace("children's", "children")
    .replace("sci fi", "science fiction")
    .replace("sci-fi", "science fiction")
    .str.title()
)

loans["genre"].value_counts()

genre
Crime              163
Children           135
Fantasy             90
Science Fiction     90
Romance             86
History             59
Name: count, dtype: int64

# Task 3

In [101]:
loans["returned"].value_counts()

returned
Y      131
Yes    125
YES    124
y      113
yes    100
N        8
NO       7
No       6
n        6
no       3
Name: count, dtype: int64

In [111]:
loans["returned"].str.strip().str.lower().value_counts()

returned
yes    349
y      244
no      16
n       14
Name: count, dtype: int64

In [112]:
loans["returned"].str.strip().str.lower().map({"yes": True, "y": True, "no": False, "n": False}).value_counts()

returned
True     593
False     30
Name: count, dtype: int64

## One step cleaning for yes/no

In [113]:
print("Missing before", loans['returned'].isna().sum())

loans["returned"] = (
    loans["returned"]
    .str.strip()
    .str.lower()
    .map({"yes": True, "y": True, "no": False, "n": False})
)

print("Missing after", loans['returned'].isna().sum())

loans["returned"].value_counts()

Missing before 0
Missing after 0


returned
True     593
False     30
Name: count, dtype: int64

# Task 4

In [ ]:
# Float conversion right away does not work:
loans["late_fee"].astype(float)

In [114]:
# Testing the steps
loans["late_fee"].str.strip().head().tolist()
loans["late_fee"].str.strip().str.replace(" kr", "").head().tolist()
loans["late_fee"].str.strip().str.replace(" kr", "").str.replace(",", ".").head().tolist()

['7.50', '0.00', '0.00', '0.00', '0.00']

## One step price cleaning

In [115]:
loans["late_fee"] = (
    loans["late_fee"]
    .str.strip()
    .str.replace("kr", "")
    .str.strip()
    .str.replace(",", ".")
).astype(float)

In [ ]:
print("Total", loans["late_fee"].sum())
loans["late_fee"].describe().round(2)

Total 7677.5


count    623.00
mean      12.32
std       16.11
min        0.00
25%        0.00
50%        0.00
75%       25.00
max       47.50
Name: late_fee, dtype: float64

# Task 5

In [116]:
dates = pd.to_datetime(loans["loan_date"], errors="coerce")

print("NaT before", loans["loan_date"].isna().sum())
print("NaT after", dates.isna().sum())

loans["loan_date"] = dates

na_dates = loans.loc[dates.isna()]
na_dates

NaT before 0
NaT after 3


,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned,notes
477,L-30463,M0007,Blue Moon Protocol,Science Fiction,NaT,2026-05-24,0.0,True,NaN
483,L-30468,M0018,Blue Moon Protocol,Science Fiction,NaT,2026-06-08,0.0,True,NaN
592,L-30572,M0090,Mimmi and the Moon,Children,NaT,2026-06-26,0.0,True,NaN


In [117]:
print(dates.dt.to_period("M").value_counts().sort_index())
print("Sum", dates.dt.to_period("M").value_counts().sum())

loan_date
2026-01     96
2026-02    100
2026-03    111
2026-04    106
2026-05    108
2026-06     99
Freq: M, Name: count, dtype: int64
Sum 620


# Task 6

In [118]:
print("Duplicates in raw data", loans_raw.duplicated().sum())
print("Duplicates in cleaned data", loans.duplicated().sum())
print("Duplicates in cleaned data with subset loan_id", loans.duplicated(subset="loan_id").sum())

Duplicates in raw data 12
Duplicates in cleaned data 18
Duplicates in cleaned data with subset loan_id 18


In [ ]:
print("Duplicates before cleaning")
loans_raw.loc[loans_raw.duplicated()].sort_index()

Duplicates before cleaning


,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned,notes
3,L-30003,M0143,The Glass Mage,fantasy,2026-01-05,2026-01-26,"0,00 kr",YES,NaN
30,L-30029,M0097,Cold Case 1987,Crime,2026-01-13,2026-01-17,"0,00 kr",YES,NaN
71,L-30068,M0006,A Quiet Witness,Crime,2026-01-24,2026-03-05,"47,50 kr",y,NaN
124,L-30119,M0024,The Dragon Who Was Afraid,children,2026-02-09,2026-03-18,"40,00 kr",YES,NaN
231,L-30223,M0007,Forest Friends,Children,2026-03-11,2026-04-02,"2,50 kr",Yes,NaN
312,L-30301,M0010,Salt and Iron,history,2026-04-03,2026-04-22,"0,00kr",YES,NaN
362,L-30350,M0019,Letters from Lisbon,romance,2026-04-16,2026-05-25,"45,00 kr",YES,NaN
482,L-30467,M0118,The Long Road to Uppsala,history,2026-05-21,2026-06-12,"2,50kr",Y,NaN
486,L-30470,M0116,Forest Friends,Children's,2026-05-21,2026-05-27,"0,00 kr",Yes,NaN
591,L-30571,M0008,Cold Case 1987,Crime,2026-06-19,2026-06-26,"0,00kr",y,NaN


In [ ]:
print("Duplicates after cleaning")
loans.loc[loans.duplicated()].sort_index()

Duplicates after cleaning


,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned,notes
3,L-30003,M0143,The Glass Mage,Fantasy,2026-01-05,2026-01-26,0.0,True,NaN
30,L-30029,M0097,Cold Case 1987,Crime,2026-01-13,2026-01-17,0.0,True,NaN
38,L-30036,M0090,Mimmi and the Moon,Children,2026-01-15,2026-02-11,15.0,True,NaN
71,L-30068,M0006,A Quiet Witness,Crime,2026-01-24,2026-03-05,47.5,True,NaN
100,L-30096,M0068,Blue Moon Protocol,Science Fiction,2026-02-01,2026-03-06,30.0,True,NaN
124,L-30119,M0024,The Dragon Who Was Afraid,Children,2026-02-09,2026-03-18,40.0,True,NaN
140,L-30134,M0028,The Glass Mage,Fantasy,2026-02-13,2026-03-25,47.5,True,NaN
219,L-30212,M0141,Between the Lines,Romance,2026-03-08,2026-03-16,0.0,True,NaN
231,L-30223,M0007,Forest Friends,Children,2026-03-11,2026-04-02,2.5,True,NaN
312,L-30301,M0010,Salt and Iron,History,2026-04-03,2026-04-22,0.0,True,NaN


In [ ]:
total_with = loans["late_fee"].sum()

loans = loans.drop_duplicates().reset_index(drop=True)
total_without = loans["late_fee"].sum()

print("Difference in kr", total_with - total_without)
print("Difference in percentage", total_without / total_with * 100)

7447.5 7677.5
